# STATIC_SIMULATION

Static simulation of drift exceedance detection (Section 3.5), based on older notebook, revamped for reviewer edits. Reads the prediction files from NONLINEAR_MODELS and Colab NN notebook and the 10 Hz fidelity series.

- Ground truth: anchored exceedance markers, same as previous.
- Decision threshold (target_decision_threshold): REXD prediction >= tau; RSTDEV prediction >= tau / (2 * 1.96). Gaussian and Chebyshev exceedance values are reported for RSTDEV as interpretation only.
- Each pred issued at t is scored against markers inside its own label window (t + start, t + end]. (times come from the pred files)
- Every config is simulated on the val and test subsets for every tolerance in TOLERANCES and saved. NOTHING is ranked or selected here. PAPER_RESULTS selects on val (tau = 0.10) and only then looks at test.

## 1. Imports

In [1]:
import json
from datetime import datetime, timezone
from functools import lru_cache
from pathlib import Path

import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import erfc
# from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score, roc_curve
from sklearn.metrics import auc, precision_recall_curve, roc_auc_score, roc_curve

## 2. Configuration

In [2]:
MANIFEST_PATH = Path("data/processed/manifest.json")
PREDICTION_DIR = Path("results/predictions")
RESULTS_DIR = Path("results/simulation")
IMAGES_PATH = Path("images/simulation")
#LOCKED_SELECTION_PATH = RESULTS_DIR / "locked_selection.json"

#EXCEED_THRESHOLDS = [0.05, 0.075, 0.10, 0.125, 0.15] # sensitivity vals (fidelity units)
TOLERANCES = [0.05, 0.075, 0.10, 0.125, 0.15] # sensitivity vals (fidelity units)
#PRIMARY_EXCEED_THRESHOLD = 0.10 # original EXCEED_THRESH
NOMINAL_TOLERANCE = 0.10 # original EXCEED_THRESH
#RSTDEV_SIGMA_FACTOR = 4.0 # RSTDEV alarm: sigma_hat >= tol / factor (original heuristic)
SPLITS = ["validation", "test"]
EVENT_MERGE_GAP_S = 60 # markers closer than this belong to one event (event-level metrics)

#SELECTION_METRIC = "pr_auc" # original criterion (PR AUC), now on val data
#SELECTION_FAMILIES = ["nonlinear", "nn"] # model families for operational lock
#OVERWRITE_EXISTING_LOCK = False # keep false once the test set has been evaluated

# Optional "delayed" comparison: also evaluate everything starting 2 h into the period. Originally used because NN predictions began 2 h late, no longer needed now that sequences use earlier rows.
EVALUATE_DELAYED = False
DELAY_S = 7200
#RUN_PART_A_VALIDATION = True
#RUN_PART_B_TEST = True
PLOT_CONFIGS = [] # optional figures ["2MIN__REXD_FWD_1200_DSMAX__rnn_baseline"], paper figs are made in PAPER_RESULTS
PLOT_SPLIT = "validation"
PLOT_FIDELITY_DECIMATE = 1

DELAYS = (0, DELAY_S) if EVALUATE_DELAYED else (0,)

mpl.rc("axes", labelsize=14)
mpl.rc("xtick", labelsize=12)
mpl.rc("ytick", labelsize=12)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_PATH.mkdir(parents=True, exist_ok=True)

manifest = json.loads(MANIFEST_PATH.read_text())
TZ = manifest["display_timezone"]
NIGHTS_MS = [(a * 1000, b * 1000) for a, b in manifest["night_periods_utc_s"]]
MODEL_NAMES = {"xgb": "XGBoost", "svr": "Support Vector Regressor", "rf": "Random Forest", "mlp": "Multilayer Perceptron", "rbfnn": "Radial Basis Function NN", "rnn": "Recurrent Neural Network", "lstm": "Long Short-Term Memory", "gru": "Gated Recurrent Unit", "daylight": "Daylight baseline", "dummy": "Dummy Regressor"}

fid = pd.read_csv(manifest["files"]["fidelity_10hz"], usecols=["current_time", "Fidelity"]).dropna()
FID_T = fid["current_time"].to_numpy("int64")
FID_F = fid["Fidelity"].to_numpy(float)

## 3. Ground truth: anchored exceedance markers and events

In [3]:
@lru_cache(maxsize=None)
def anchored_exceedance_markers(t0_ms, t1_ms, thresh):
    """Marker times in [t0, t1]: |F - F_anchor| > thresh, anchor resets at each marker (original rule)."""
    sel = (FID_T >= t0_ms) & (FID_T <= t1_ms)
    times, values = FID_T[sel], FID_F[sel]
    mark = np.zeros(len(values), dtype=bool)
    if len(values):
        anchor = values[0]
        for i in range(1, len(values)):
            if abs(values[i] - anchor) > thresh:
                mark[i] = True
                anchor = values[i]
    return times[mark]


def group_events(marker_times):
    """Consecutive markers closer than EVENT_MERGE_GAP_S form one event (start = first marker)."""
    if len(marker_times) == 0:
        return np.array([], dtype="int64")
    new = np.r_[True, np.diff(marker_times) > EVENT_MERGE_GAP_S * 1000]
    return marker_times[new]

## 4. Interpretation layer and evaluation

In [4]:
Z_975 = 1.959963985 # used excel's norm.s.inv for this calculation because that's how I learned to calculate the value from Mrs. S-K's class. There's probably a library for this


def target_decision_threshold(target_type, tolerance):
    if target_type == "REXD":
        return tolerance
    elif target_type == "RSTDEV":
        return tolerance / (2 * Z_975)


def pr_auc_score(labels, scores):
    # area under the PR curve (trapezoidal), as in the paper
    precision, recall, _ = precision_recall_curve(labels, scores)
    return auc(recall, precision)


def alarm_threshold(metric, tol):
    #if metric == "RSTDEV":
    #    return tol / RSTDEV_SIGMA_FACTOR # operational used by the original sim script
    #if metric == "REXD":
    #    return tol
    if metric in ("REXD", "RSTDEV"):
        return target_decision_threshold(metric, tol)
    return 0.5 # binary baselines


def rstdev_interpretation(sigma, tol):
    """Supplementary interpretation of a predicted standard deviation (not used for alarms)."""
    s = np.clip(np.asarray(sigma, float), 1e-12, None)
    return {"gaussian_p_exceed": erfc(tol / (s * np.sqrt(2.0))), "chebyshev_bound": np.minimum(1.0, (s / tol) ** 2)}


def daylight_baseline(template):
    """0 when the target window starts at night, 1 in daylight (external sunrise/sunset data)."""
    base = template.copy()
    start = base["current_time"].to_numpy() + base["window_start_offset_s"].to_numpy() * 1000
    night = np.zeros(len(base), dtype=bool)
    for a, b in NIGHTS_MS:
        night |= (start >= a) & (start < b)
    base["y_pred"] = (~night).astype(float)
    base["model"], base["model_family"], base["metric"] = "daylight_baseline", "baseline", "BINARY"
    return base


def evaluate(pred, tol, delay_s=0):
    """Interval- and event-level evaluation of one prediction frame for one exceedance threshold."""
    pred = pred.dropna(subset=["y_pred"]).sort_values("current_time")
    if delay_s:
        pred = pred[pred["current_time"] >= pred["current_time"].min() + delay_s * 1000]
    t = pred["current_time"].to_numpy("int64")
    start = t + pred["window_start_offset_s"].to_numpy() * 1000
    end = t + pred["window_end_offset_s"].to_numpy() * 1000
    markers = anchored_exceedance_markers(int(t.min()), int(end.max()), tol)
    events = group_events(markers)
    labels = (np.searchsorted(markers, end, side="right") - np.searchsorted(markers, start, side="right")) > 0
    scores = pred["y_pred"].to_numpy(float)
    metric = pred["metric"].iloc[0]
    flags = scores >= alarm_threshold(metric, tol)

    tp, fn = int((flags & labels).sum()), int((~flags & labels).sum())
    fp, tn = int((flags & ~labels).sum()), int((~flags & ~labels).sum())
    total = tp + fn + fp + tn
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    both = labels.any() and (~labels).any()
    hours = total * int(pred["sampling_s"].iloc[0]) / 3600

    # alarm episodes (consecutive alarms) and false alarm episodes (no marker in any covered window)
    step_ms = int(pred["sampling_s"].iloc[0]) * 1000
    alarm_idx = np.where(flags)[0]
    episodes = np.split(alarm_idx, np.where(np.diff(t[alarm_idx]) > 1.5 * step_ms)[0] + 1) if len(alarm_idx) else []
    false_eps = sum(1 for ep in episodes if not labels[ep].any())

    # event level: an event at te is detected if an alarm issued at t_a has te in (t_a + start, t_a + end]
    detected, leads, evaluable = [], [], 0
    for te in events:
        covering = (start < te) & (te <= end)
        if not covering.any():
            continue
        evaluable += 1
        hit = covering & flags
        detected.append(bool(hit.any()))
        if hit.any():
            leads.append((te - t[hit].min()) / 1000)
    return {
        #"exceed_threshold": tol, "alarm_threshold": alarm_threshold(metric, tol), "evaluation_delay_s": delay_s,
        "tolerance": tol, "decision_threshold": alarm_threshold(metric, tol), "evaluation_delay_s": delay_s,
        "n_predictions": total, "hours": hours, "positives": int(labels.sum()), "prevalence": float(labels.mean()) if total else np.nan,
        "tp": tp, "fn": fn, "fp": fp, "tn": tn, "accuracy": (tp + tn) / total if total else np.nan,
        "recall": recall, "precision": precision, "f1": 2 * precision * recall / (precision + recall) if precision + recall else 0.0,
        "false_positive_rate": fp / (fp + tn) if fp + tn else np.nan,
        "roc_auc": roc_auc_score(labels, scores) if both else np.nan,
        # "pr_auc": average_precision_score(labels, scores) if both else np.nan,
        "pr_auc": pr_auc_score(labels, scores) if both else np.nan,
        "false_positive_predictions_per_hour": fp / hours if hours else np.nan,
        "alarm_episodes": len(episodes), "false_alarm_episodes": false_eps,
        "false_alarm_episodes_per_hour": false_eps / hours if hours else np.nan,
        "events": len(events), "events_evaluable": evaluable,
        "event_detection_rate": float(np.mean(detected)) if detected else np.nan,
        "missed_event_rate": 1 - float(np.mean(detected)) if detected else np.nan,
        "median_warning_lead_s": float(np.median(leads)) if leads else np.nan,
        "min_warning_lead_s": float(np.min(leads)) if leads else np.nan,
    }, {"t": t, "labels": labels, "scores": scores, "flags": flags, "markers": markers}


# def describe(pred):
def describe(pred, split=None):
    first = pred.iloc[0]
    #return {"sampling": first["sampling"], "target": first["target"], "model": first["model"], "model_family": first["model_family"], "metric": first["metric"], "lead_s": first["lead_s"]}
    return {"config_id": f"{first['sampling']}__{first['target']}__{first['model']}", "split": split, "sampling": first["sampling"], "target": first["target"], "model": first["model"], "model_family": first["model_family"], "metric": first["metric"], "lead_s": first["lead_s"], "target_type": first["target"].split("_")[0], "window_samples": first["window_samples"], "variant": first["variant"], "savgol_filtered": first["variant"] == "SAVGOL"}


def load_split(path, split):
    df = pd.read_csv(path)
    return df[df["split"] == split]

## 5. Plotting (original simulator figure and ROC / PR curves)

In [5]:
def local(ms):
    t = pd.to_datetime(ms, unit="ms", utc=True)
    return t.dt.tz_convert(TZ) if isinstance(t, pd.Series) else t.tz_convert(TZ)


def plot_simulation(pred, detail, tol, fig_id):
    info = describe(pred)
    t0, t1 = int(detail["t"].min()), int(detail["t"].max() + pred["window_end_offset_s"].max() * 1000)
    sel = (FID_T >= t0) & (FID_T <= t1)
    fig, ax1 = plt.subplots(figsize=(12, 5))
    ax1.plot(local(FID_T[sel][::PLOT_FIDELITY_DECIMATE]), FID_F[sel][::PLOT_FIDELITY_DECIMATE], label="Fidelity (0.1s)", linewidth=1)
    ax1.plot(local(detail["markers"]), np.interp(detail["markers"], FID_T, FID_F), linestyle="None", marker="x", label=f"Fidelity exceedance (>{tol})")
    ax1.set_ylabel("Polarization Fidelity")
    ax1.grid(alpha=0.3)
    
    for a, b in NIGHTS_MS:
        if a < t1 and b > t0:
            ax1.axvspan(local(max(a, t0)), local(min(b, t1)), alpha=0.3, color="gray", zorder=0)
            
    ax2 = ax1.twinx()
    ax2.plot(local(detail["t"]), detail["scores"], color="red", linewidth=2, label=f"Predicted ({int(pred['sampling_s'].iloc[0])}s cadence)")
    thr = alarm_threshold(info["metric"], tol)
    #ax2.axhline(thr, linestyle="--", linewidth=1.5, label=f"StDev alarm threshold (tol/{RSTDEV_SIGMA_FACTOR:g} = {thr:g})" if info["metric"] == "RSTDEV" else f"Threshold ({thr:g})")
    ax2.axhline(thr, linestyle="--", linewidth=1.5, label=f"Decision threshold (tau/(2*1.96) = {thr:.4g})" if info["metric"] == "RSTDEV" else f"Decision threshold (tau = {thr:g})") # just abbreviate z here
    ax2.set_ylabel(f"{info['metric']} Prediction")
    ax1.xaxis.set_major_locator(mdates.HourLocator(byhour=range(0, 24, 2), tz=TZ))
    ax1.xaxis.set_major_formatter(mdates.DateFormatter("%d %b\n%H%M", tz=TZ))
    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, loc="upper left")
    plt.tight_layout()
    plt.savefig(IMAGES_PATH / f"{fig_id}.png", dpi=300)
    plt.show()


def plot_roc_pr(detail, title, fig_id):
    y, s = detail["labels"], detail["scores"]
    
    if not (y.any() and (~y).any()):
        print("ROC/PR undefined (single class).")
        return
        
    fpr, tpr, _ = roc_curve(y, s)
    precision, recall, _ = precision_recall_curve(y, s)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f"ROC and PR Curves {title}", fontsize=15)
    ax1.plot(fpr, tpr, label=f"AUC={roc_auc_score(y, s):.4f}")
    ax1.plot([0, 1], [0, 1], "--", label="Random")
    ax1.set_xlabel("False Positive Rate")
    ax1.set_ylabel("True Positive Rate")
    ax1.set_title("ROC Curve")
    # ax2.plot(recall, precision, label=f"AP={average_precision_score(y, s):.4f}")
    ax2.plot(recall, precision, label=f"PR AUC={auc(recall, precision):.4f}")
    ax2.hlines(y.mean(), 0, 1, linestyles="--", label="Baseline")
    ax2.set_xlabel("Recall")
    ax2.set_ylabel("Precision")
    ax2.set_title("Precision-Recall Curve")
    for ax in (ax1, ax2):
        ax.legend()
        ax.grid(alpha=0.3)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(IMAGES_PATH / f"{fig_id}.png", dpi=300)
    plt.show()

## 6. Simulation of every configuration on val and test

Every config is simulated on its val rows and on its test rows for every tolerance. The daylight baseline is added once per (sampling, target). Results go to results/simulation/simulator_results_validation.csv and simulator_results_test.csv. Model selection (validation PR AUC at tau = 0.10) is done in PAPER_RESULTS, NOT here.

In [6]:
'''if RUN_PART_A_VALIDATION:
     val_rows = []
     for path in sorted((PREDICTION_DIR / "validation").glob("*.csv")):
         pred = load_split(path, "validation")
         if pred.empty:
             continue
         for tol in EXCEED_THRESHOLDS:
             res, _ = evaluate(pred, tol)
             val_rows.append({**describe(pred), **res})
     validation_sim = pd.DataFrame(val_rows)
     validation_sim.to_csv(RESULTS_DIR / "validation_simulation_results.csv", index=False)
     primary_val = validation_sim[(validation_sim["exceed_threshold"] == PRIMARY_EXCEED_THRESHOLD) & (validation_sim["lead_s"] == 0)]
     display(primary_val.sort_values(SELECTION_METRIC, ascending=False).head(20))

'''
sim_rows = []
for split in SPLITS:
    baseline_done = set()
    files = sorted((PREDICTION_DIR / split).glob("*.csv"))
    for k, path in enumerate(files):
        pred = load_split(path, split)
        if pred.empty:
            continue
        frames = [pred]
        key = (pred["sampling"].iloc[0], pred["target"].iloc[0])
        if key not in baseline_done:
            baseline_done.add(key)
            frames.append(daylight_baseline(pred))
        for frame in frames:
            for tol in TOLERANCES:
                res, _ = evaluate(frame, tol)
                sim_rows.append({**describe(frame, split), **res})
        if (k + 1) % 200 == 0:
            print(f"{split}: {k + 1}/{len(files)}")

sim_results = pd.DataFrame(sim_rows)
for split in SPLITS:
    out = sim_results[sim_results["split"] == split]
    out.to_csv(RESULTS_DIR / f"simulator_results_{split}.csv", index=False)
    print(f"{split}: {out['config_id'].nunique()} configurations x {len(TOLERANCES)} tolerances -> simulator_results_{split}.csv")

validation: 200/1224
validation: 400/1224
validation: 600/1224
validation: 800/1224
validation: 1000/1224
validation: 1200/1224
test: 200/1224
test: 400/1224
test: 600/1224
test: 800/1224
test: 1000/1224
test: 1200/1224
validation: 1296 configurations x 5 tolerances -> simulator_results_validation.csv
test: 1296 configurations x 5 tolerances -> simulator_results_test.csv


In [7]:
'''# selection moved to PAPER_RESULTS
if RUN_PART_A_VALIDATION:
    if LOCKED_SELECTION_PATH.exists() and not OVERWRITE_EXISTING_LOCK:
        lock = json.loads(LOCKED_SELECTION_PATH.read_text())
        print("Existing lock kept (OVERWRITE_EXISTING_LOCK = False).")
    else:
        eligible = primary_val[primary_val["model_family"].isin(SELECTION_FAMILIES)].dropna(subset=[SELECTION_METRIC])
        chosen = eligible.sort_values(SELECTION_METRIC, ascending=False).iloc[0]
        meta = next(t for t in manifest["targets"] if t["name"] == chosen["target"])
        lock = {"sampling": chosen["sampling"], "target": chosen["target"], "model": chosen["model"],
                "model_family": chosen["model_family"], "metric": meta["metric"], "window_samples": meta["window_samples"],
                "variant": meta["variant"], "lead_s": meta["lead_s"],
                "window_start_offset_s": meta["window_start_offset_s"], "window_end_offset_s": meta["window_end_offset_s"],
                "exceed_threshold": PRIMARY_EXCEED_THRESHOLD, "rstdev_sigma_factor": RSTDEV_SIGMA_FACTOR,
                "event_merge_gap_s": EVENT_MERGE_GAP_S, "selection_metric": SELECTION_METRIC,
                "selection_value_validation": float(chosen[SELECTION_METRIC]), "n_candidates": int(len(eligible)),
                "created_utc": datetime.now(timezone.utc).isoformat(),
                "note": "Selected from validation-period simulation only; the test period was not used."}
        LOCKED_SELECTION_PATH.write_text(json.dumps(lock, indent=2))
        print(f"Wrote {LOCKED_SELECTION_PATH}")
    print(json.dumps(lock, indent=2))
    '''

'# selection moved to PAPER_RESULTS\nif RUN_PART_A_VALIDATION:\n    if LOCKED_SELECTION_PATH.exists() and not OVERWRITE_EXISTING_LOCK:\n        lock = json.loads(LOCKED_SELECTION_PATH.read_text())\n        print("Existing lock kept (OVERWRITE_EXISTING_LOCK = False).")\n    else:\n        eligible = primary_val[primary_val["model_family"].isin(SELECTION_FAMILIES)].dropna(subset=[SELECTION_METRIC])\n        chosen = eligible.sort_values(SELECTION_METRIC, ascending=False).iloc[0]\n        meta = next(t for t in manifest["targets"] if t["name"] == chosen["target"])\n        lock = {"sampling": chosen["sampling"], "target": chosen["target"], "model": chosen["model"],\n                "model_family": chosen["model_family"], "metric": meta["metric"], "window_samples": meta["window_samples"],\n                "variant": meta["variant"], "lead_s": meta["lead_s"],\n                "window_start_offset_s": meta["window_start_offset_s"], "window_end_offset_s": meta["window_end_offset_s"],\n       

### 6.1 Optional figures (nominal tolerance only)

Figures for any config listed in PLOT_CONFIGS. The figures for the selected models are made in PAPER_RESULTS.

In [8]:
'''def lead_variant_files(lock, split_dir):
    files = []
    for t in manifest["targets"]:
        base = next(x for x in manifest["targets"] if x["name"] == lock["target"])["base"]
        if t["base"] == base:
            p = PREDICTION_DIR / split_dir / f"{lock['sampling']}__{t['name']}__{lock['model']}.csv"
            if p.exists():
                files.append((t["lead_s"], p))
    return sorted(files)

if RUN_PART_A_VALIDATION:
    rows = []
    for lead, path in lead_variant_files(lock, "validation"):
        pred = load_split(path, "validation")
        for frame in (pred, daylight_baseline(pred)):
            for delay in DELAYS:
                res, detail = evaluate(frame, lock["exceed_threshold"], delay)
                rows.append({**describe(frame), **res})
        if lead == 0:
            res, detail = evaluate(pred, lock["exceed_threshold"])
            plot_simulation(pred, detail, lock["exceed_threshold"], f"validation_simres_{lock['sampling']}_{lock['target']}_{lock['model']}")
    display(pd.DataFrame(rows))
'''
for config_id in PLOT_CONFIGS:
    pred = load_split(PREDICTION_DIR / PLOT_SPLIT / f"{config_id}.csv", PLOT_SPLIT)
    res, detail = evaluate(pred, NOMINAL_TOLERANCE)
    plot_simulation(pred, detail, NOMINAL_TOLERANCE, f"{PLOT_SPLIT}_simres_{config_id}")
    plot_roc_pr(detail, f"({config_id}, {PLOT_SPLIT})", f"{PLOT_SPLIT}_roc_pr_{config_id}")

### (old Part B removed: test evaluation of the selected models is in PAPER_RESULTS)

In [9]:
'''
if RUN_PART_B_TEST and not LOCKED_SELECTION_PATH.exists():
    print("No locked selection yet: run Part A first.")
elif RUN_PART_B_TEST:
    lock = json.loads(LOCKED_SELECTION_PATH.read_text())
    test_files = lead_variant_files(lock, "test")
    if not test_files:
        print("Locked test preds not found: run NONLINEAR_MODELS (or the Colab NN notebook) first.")
    else:
        rows, locked_pred, locked_detail = [], None, None
        for lead, path in test_files:
            pred = load_split(path, "test")
            for frame in (pred, daylight_baseline(pred)):
                for tol in EXCEED_THRESHOLDS:
                    for delay in DELAYS:
                        res, detail = evaluate(frame, tol, delay)
                        rows.append({**describe(frame), **res})
                        if lead == 0 and frame is pred and tol == lock["exceed_threshold"] and delay == 0:
                            locked_pred, locked_detail = pred, detail
        test_sim = pd.DataFrame(rows)
        test_sim.to_csv(RESULTS_DIR / "test_simulation_results.csv", index=False)
        display(test_sim[(test_sim["exceed_threshold"] == lock["exceed_threshold"]) & (test_sim["evaluation_delay_s"] == 0)])

        if locked_pred is not None:
            title = (f"({MODEL_NAMES.get(lock['model'].split('_')[0], lock['model'])}, {lock['metric']} (w={lock['window_samples']}, {lock['variant']}), {lock['sampling']})")
            plot_simulation(locked_pred, locked_detail, lock["exceed_threshold"], f"simres_test_{lock['sampling']}_{lock['target']}_{lock['model']}")
            plot_roc_pr(locked_detail, title, f"test_{lock['sampling']}_{lock['target']}_{lock['model']}_roc_pr")
            if lock["metric"] == "RSTDEV":
                interp = pd.DataFrame(rstdev_interpretation(locked_detail["scores"], lock["exceed_threshold"]))
                interp.insert(0, "current_time", locked_detail["t"])
                interp["sigma_hat"] = locked_detail["scores"]
                interp["alarm (sigma_hat >= tol/factor)"] = locked_detail["flags"]
                interp.to_csv(RESULTS_DIR / "test_rstdev_interpretation.csv", index=False)
                display(interp.describe())
'''

'\nif RUN_PART_B_TEST and not LOCKED_SELECTION_PATH.exists():\n    print("No locked selection yet: run Part A first.")\nelif RUN_PART_B_TEST:\n    lock = json.loads(LOCKED_SELECTION_PATH.read_text())\n    test_files = lead_variant_files(lock, "test")\n    if not test_files:\n        print("Locked test preds not found: run NONLINEAR_MODELS (or the Colab NN notebook) first.")\n    else:\n        rows, locked_pred, locked_detail = [], None, None\n        for lead, path in test_files:\n            pred = load_split(path, "test")\n            for frame in (pred, daylight_baseline(pred)):\n                for tol in EXCEED_THRESHOLDS:\n                    for delay in DELAYS:\n                        res, detail = evaluate(frame, tol, delay)\n                        rows.append({**describe(frame), **res})\n                        if lead == 0 and frame is pred and tol == lock["exceed_threshold"] and delay == 0:\n                            locked_pred, locked_detail = pred, detail\n        t